Using the DATA_GOV_API_KEY, retrieve data from `https://data.sfgov.org/resource/wg3w-h783`, SF Police Department Report Data.

complete store_to_gcs() to store retrieved data to GCS, and retrieve_data_from_gcs() to retrieve data from GCS.
- store_to_gcs(service_account_key, project_id, bucket_name, file_name, data) : upload data as file_name on the given gcs.
- retrieve_data_from_gcs(service_account_key, project_id, bucket_name, file_name, key_list) : return a list of list which includes values corresponding to keys in key_list.
If 
```
key_list = ['incident_datetime', 'report_datetime', 'incident_code',
            'incident_category', 'incident_description', 'latitude',
            'longitude', 'police_district']
```
This should return
```
[['2025-08-26T23:17:00.000',
  '2025-08-26T23:17:00.000',
  '07041',
  'Recovered Vehicle',
  'Vehicle, Recovered, Auto',
  None,
  None,
  'Out of SF'],
 ['2025-08-27T00:37:00.000',
  '2025-08-27T00:37:00.000',
  '04134',
  'Assault',
  'Battery',
  '37.78041458129883',
  '-122.44901275634766',
  'Park'],....]
```
If a key doesn't exist, its value should be `None`

In [12]:
import datetime
import json
import os 

from dotenv import load_dotenv
from google.oauth2 import service_account
from google.cloud import storage
import requests

In [31]:
load_dotenv(dotenv_path="/Users/lokeshmuvva/Documents/msds692_data_acquisition_2025/Day3/.env")

api_key = os.getenv("DATA_GOV_API_KEY")

print(api_key)

rxhKllwEmTsqiYEVlBIxFMZBYpCCPlsbu8EWS744


- Typically authorization in a header uses the format of..
```
headers = {"Authorization": f"Bearer {api_key}", "Content-Type": "application/json"}
```
However, based on the [documentaiton](https://api.data.gov/docs/developer-manual/), I followed the following format.

Also, the data specific meta data and details can be searchable. (Ex.[link](https://data.sfgov.org/Public-Safety/Police-Department-Incident-Reports-2018-to-Present/wg3w-h783/about_data))

In [37]:
url = 'https://data.sfgov.org/resource/wg3w-h783'
header = {'X-Api-Key': api_key}

response = requests.get(url, headers=header)

In [38]:
data = response.json()

print(data)

[{'row_id': '148998204134', 'incident_datetime': '2025-06-13T12:41:00.000', 'incident_date': '2025-06-13T00:00:00.000', 'incident_time': '12:41', 'incident_year': '2025', 'incident_day_of_week': 'Friday', 'report_datetime': '2025-06-13T12:46:00.000', 'incident_id': '1489982', 'incident_number': '250329888', 'cad_number': '251641497', 'report_type_code': 'II', 'report_type_description': 'Initial', 'incident_code': '04134', 'incident_category': 'Assault', 'incident_subcategory': 'Simple Assault', 'incident_description': 'Battery', 'resolution': 'Open or Active', 'intersection': 'JOHN F SHELLEY DR \\ MANSELL ST', 'cnn': '20872000', 'police_district': 'Ingleside', 'analysis_neighborhood': 'McLaren Park', 'supervisor_district': '9', 'supervisor_district_2012': '9', 'latitude': '37.7181282043457', 'longitude': '-122.41417694091797', 'point': {'type': 'Point', 'coordinates': [-122.414176941, 37.718128204]}, 'data_as_of': '2025-06-14T09:37:39.000', 'data_loaded_at': '2025-06-15T09:53:25.000', 

Create store_to_gcs(service_account_key, project_id, bucket_name, file_name, data) which upload data as file_name on the given gcs.

In [39]:
def store_to_gcs(service_account_key: str,
                 project_id: str,
                 bucket_name: str,
                 file_name: str,
                 data: str) -> None:    
    credentials = service_account.Credentials.from_service_account_file(service_account_key)
    client = storage.Client(project=project_id,
                            credentials=credentials)
    bucket = client.bucket(bucket_name)
    file = bucket.blob(file_name)
    file.upload_from_string(data)

In [40]:
service_account_key = os.getenv("GCP_SERVICE_ACCOUNT_KEY")
project_id = os.getenv("GCP_PROJECT_ID")
bucket_name = os.getenv("GCP_BUCKET_NAME")
file_name = f"sf_police_report/{datetime.date.today()}.json"
store_to_gcs(service_account_key,
            project_id,
            bucket_name,
            file_name,
            json.dumps(data, indent=4))

Complete retrieve_data_from_gcs() to return a list of list which includes values corresponding to keys in key_list.
If 
```
key_list = ['incident_datetime', 'report_datetime', 'incident_code',
            'incident_category', 'incident_description', 'latitude',
            'longitude', 'police_district']
```
This should return
```
[['2025-08-26T23:17:00.000',
  '2025-08-26T23:17:00.000',
  '07041',
  'Recovered Vehicle',
  'Vehicle, Recovered, Auto',
  None,
  None,
  'Out of SF'],
 ['2025-08-27T00:37:00.000',
  '2025-08-27T00:37:00.000',
  '04134',
  'Assault',
  'Battery',
  '37.78041458129883',
  '-122.44901275634766',
  'Park'],....]
```
If a key doesn't exist, its value should be `None`

In [44]:
def retrieve_data_from_gcs(service_account_key: str,
                           project_id: str,
                           bucket_name: str,
                           file_name: str,
                           key_list: list
                           ) -> list:
    credentials = service_account.Credentials.from_service_account_file(service_account_key)
    client = storage.Client(project=project_id,
                            credentials=credentials)
    bucket = client.bucket(bucket_name)
    file = bucket.blob(file_name)
    content = json.loads(file.download_as_string())

    output = []
    for data in content:
        row = []

        for key in key_list:
            row.append(data.get(key, None))
        output.append(row)
    return output

In [45]:
key_list = ['incident_datetime', 'report_datetime', 'incident_code',
            'incident_category', 'incident_description', 'latitude',
            'longitude', 'police_district']
data = retrieve_data_from_gcs(service_account_key, 
                              project_id,
                              bucket_name,
                              f"sf_police_report/{datetime.date.today()}.json",
                              key_list)

In [47]:
data[:6]

[['2025-06-13T12:41:00.000',
  '2025-06-13T12:46:00.000',
  '04134',
  'Assault',
  'Battery',
  '37.7181282043457',
  '-122.41417694091797',
  'Ingleside'],
 ['2025-05-21T00:00:00.000',
  '2025-05-21T08:15:00.000',
  '06224',
  'Larceny Theft',
  'Theft, From Unlocked Vehicle, >$950',
  None,
  None,
  'Ingleside'],
 ['2025-06-11T08:00:00.000',
  '2025-06-12T11:27:00.000',
  '05043',
  'Burglary',
  'Burglary, Residence, Unlawful Entry',
  '37.77315139770508',
  '-122.42222595214844',
  'Northern'],
 ['2025-06-12T00:09:00.000',
  '2025-06-12T00:09:00.000',
  '03014',
  'Robbery',
  'Robbery, Street or Public Place, W/ Force',
  '37.761837005615234',
  '-122.41935729980469',
  'Mission'],
 ['2025-05-23T00:00:00.000',
  '2025-06-12T12:24:00.000',
  '09027',
  'Fraud',
  'False Personation',
  '37.734825134277344',
  '-122.38738250732422',
  'Bayview'],
 ['2025-06-13T02:55:00.000',
  '2025-06-13T02:55:00.000',
  '07041',
  'Recovered Vehicle',
  'Vehicle, Recovered, Auto',
  None,
  None